In [83]:
%load_ext autoreload
%autoreload 2
%env CUDA_VISIBLE_DEVICES=1

import numpy as np
import sys
import os
import matplotlib.pyplot as plt
import vis_utils

from utils.utils import get_path
from utils.io_utils import load_multiple_res, read_ripser_result
from utils.toydata_utils import get_toy_data
from utils.fig_utils import dataset_to_print, plot_dgm_loops, dist_to_print, plot_edges_on_scatter
from utils.pd_utils import sort_cycle
from utils.confidence_utils import median_max_life_time, get_bottleneck_dist
from utils.passing_cells_utils import distribution_max, distribution_split

from vis_utils.loaders import load_dataset
from vis_utils.plot import plot_scatter
from vis_utils.utils import load_dict, save_dict
from vis_utils.tsne_wrapper import TSNEwrapper

from openTSNE.affinity import Affinities

import umap
from ripser import Rips
from ripser.ripser import get_greedy_perm

from mpl_toolkits.axes_grid1 import make_axes_locatable

from matplotlib import collections  as mc
from matplotlib.colors import Normalize
import matplotlib.cm

from openTSNE.nearest_neighbors import PrecomputedNeighbors
from openTSNE.affinity import PerplexityBasedNN

from vis_utils.utils import kNN_graph, kNN_dists
from scipy.spatial.distance import pdist, squareform
import glasbey
import scipy.sparse
import networkx as nx

from sklearn.decomposition import PCA

from persim import plot_diagrams
from utils.pd_utils import get_life_times
import pandas as pd

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
env: CUDA_VISIBLE_DEVICES=1


In [84]:
root_path = get_path("data")
fig_path = get_path("figures")
seeds = [0]
k = 15

style_file = "utils.style"
plt.style.use(style_file)

distances = {
    # "euclidean": [{}],
    "diffusion": [
        {"k": 15, "t": 8, "kernel": "sknn", "include_self": False},
        {"k": 100, "t": 8, "kernel": "sknn", "include_self": False},
        {"k": 15, "t": 64, "kernel": "sknn", "include_self": False},
        {"k": 100, "t": 64, "kernel": "sknn", "include_self": False},
    ],
    "eff_res": [
        {"corrected": True, "weighted": False, "k": 15, "disconnect": True},
        {"corrected": True, "weighted": False, "k": 100, "disconnect": True},
    ],
}

In [ ]:
def get_life_geq_scale(all_res, threshold_conditions, scale):
    root_path = get_path("data")
    geq_maxs = []
    idx = 0
    for i, distance in enumerate(list(all_res.keys())):
        for j, full_dist in enumerate(all_res[distance]):
            res = all_res[distance][full_dist]
            life_array = all_res[distance][full_dist]["dgms"][1]
            reference_life_long = life_array[:,1] - life_array[:,0]
    
            geq_max = np.sum(np.array(reference_life_long) >= int(scale)* threshold_conditions[idx])
            geq_maxs.append(geq_max)
            idx+=1
            
    return geq_maxs

## embedding loading and mask making 

In [ ]:
sex_color = "male_orange"
for i in range(10):
    globals()[f"embd_10x_{sex_color}_{i}"] = np.load(os.path.join(root_path, "embd_n_mask","embd_after_procrustes", f"embd_10x_{sex_color}_split_{i}.npy"))
    print(globals()[f"embd_10x_{sex_color}_{i}"].shape)

(5289, 2)
(5289, 2)
(5289, 2)
(5289, 2)
(5289, 2)
(5289, 2)
(5289, 2)
(5289, 2)
(5289, 2)
(5288, 2)


In [ ]:
sex_color = "female_orange"
for i in range(5):
    globals()[f"embd_10x_{sex_color}_{i}"] = np.load(os.path.join(root_path, "embd_n_mask","embd_after_procrustes", f"embd_10x_{sex_color}_split_{i}.npy"))
    print(globals()[f"embd_10x_{sex_color}_{i}"].shape)

(5000, 2)
(5000, 2)
(5000, 2)
(5000, 2)
(5000, 2)


In [ ]:
_, _, _, _, full_d_male = load_dataset(root_path, "yao_10x_male", k=k)
cluster_orange_male = [cluster for i, cluster in enumerate(full_d_male["clusterNames"]) 
                  if "_Sst" in cluster or "_Pvalb" in cluster]
idx_orange = [i for i, cluster in enumerate(full_d_male["clusterNames"]) 
              if "_Sst" in cluster or "_Pvalb" in cluster]
mask_orange_male = [idx in idx_orange for idx in full_d_male['clusters']]

In [ ]:
_, _, _, _, full_d_female = load_dataset(root_path, "yao_10x_female", k=k)
cluster_orange_female = [cluster for i, cluster in enumerate(full_d_female["clusterNames"]) 
                  if "_Sst" in cluster or "_Pvalb" in cluster]
idx_orange = [i for i, cluster in enumerate(full_d_female["clusterNames"]) 
              if "_Sst" in cluster or "_Pvalb" in cluster]
mask_orange_female = [idx in idx_orange for idx in full_d_female['clusters']]

# std * n

In [ ]:
threshold_conditions = []
sex_data = [f"yao_10x_male_orange_split_{i}" for i in range(10)]+ [f"yao_10x_female_orange_split_{i}" for i in range(5)]
for full_dist in full_dists:
    bdists = [] 
    for i, dataset in enumerate(sex_data):
        bdist = load_dict(os.path.join(get_path("data"),f"{dataset}/bottleneck_dists_split_quarter_{full_dist}.pkl"))["bottleneck"]
        bdists.append(bdist)
    bdists = np.array(bdists)
    bdists_flat = bdists[np.triu_indices_from(bdists, k=1)]
    print(len(bdists_flat))
    threshold_conditions.append(np.sqrt(1/(15**2)*np.sum(bdists_flat**2)))

105
105
105
105
105
105


In [ ]:
[float("{:.8f}".format(4 * i)) for i in threshold_conditions]

[0.66588862, 0.30592074, 0.09683287, 0.01773551, 0.00317824, 6.101e-05]

In [ ]:
[float("{:.8f}".format(i)) for i in threshold_conditions]

[0.16647215, 0.07648018, 0.02420822, 0.00443388, 0.00079456, 1.525e-05]

## *4

In [ ]:
geq_maxs_fe = []
for i in range(5):
    dataset = f"yao_10x_female_orange_split_{i}"
    all_res = load_multiple_res(datasets=dataset, n=None, embd_dims=None, sigmas=None, distances=distances, seeds=0, root_path=root_path, n_threads=1)
    geq_maxs = get_life_geq_scale(all_res, threshold_conditions, 4)
    print(geq_maxs)
    geq_maxs_fe.append(geq_maxs)
    globals()[f"dist{i}"] = distribution_split(all_res, cell_group_name = dataset, clusters = cluster_orange_female, full_d = full_d_female, mask = mask_orange_female, geq_maxs = geq_maxs, total_num = 5)
    save_dict(globals()[f"dist{i}"], os.path.join(get_path("data"),"dist_bottleneck_newsplit_std_scale", f"dist_10x_female_orange_split_{i}_scale_4.pkl"))

Done with yao_10x_female_orange_split_0 None diffusion_k_15_t_8_kernel_sknn_include_self_False n_outliers=0, perturbation=None
Done with yao_10x_female_orange_split_0 None diffusion_k_100_t_8_kernel_sknn_include_self_False n_outliers=0, perturbation=None
Done with yao_10x_female_orange_split_0 None diffusion_k_15_t_64_kernel_sknn_include_self_False n_outliers=0, perturbation=None
Done with yao_10x_female_orange_split_0 None diffusion_k_100_t_64_kernel_sknn_include_self_False n_outliers=0, perturbation=None
Done with yao_10x_female_orange_split_0 None eff_res_corrected_True_weighted_False_k_15_disconnect_True n_outliers=0, perturbation=None
Done with yao_10x_female_orange_split_0 None eff_res_corrected_True_weighted_False_k_100_disconnect_True n_outliers=0, perturbation=None
[0, 2, 0, 0, 0, 0]
Done with yao_10x_female_orange_split_1 None diffusion_k_15_t_8_kernel_sknn_include_self_False n_outliers=0, perturbation=None
Done with yao_10x_female_orange_split_1 None diffusion_k_100_t_8_kern

In [ ]:
dist = {}
dist['clusters'] = dist1['clusters']
dist['idx'] = [a for i in range(5) for a in globals()[f"dist{i}"]['idx']]
dist['density'] = np.array([a for i in range(5) for a in globals()[f"dist{i}"]['density']])
save_dict(dist, os.path.join(get_path("data"),"dist_bottleneck_newsplit_std_scale", f"dist_10x_female_orange_splits_scale_4.pkl"))

In [ ]:
geq_maxs_ma = []
for i in range(10):
    dataset = f"yao_10x_male_orange_split_{i}"
    all_res = load_multiple_res(datasets=dataset, n=None, embd_dims=None, sigmas=None, distances=distances, seeds=0, root_path=root_path, n_threads=1)
    geq_maxs = get_life_geq_scale(all_res, threshold_conditions, 4)
    print(geq_maxs)
    geq_maxs_ma.append(geq_maxs)
    globals()[f"dist{i}"] = distribution_split(all_res, cell_group_name = dataset, clusters = cluster_orange_male, full_d = full_d_male, mask = mask_orange_male, geq_maxs = geq_maxs, total_num = 10)
    save_dict(globals()[f"dist{i}"], os.path.join(get_path("data"),"dist_bottleneck_newsplit_std_scale", f"dist_10x_male_orange_split_{i}_scale_4.pkl"))

Done with yao_10x_male_orange_split_0 None diffusion_k_15_t_8_kernel_sknn_include_self_False n_outliers=0, perturbation=None
Done with yao_10x_male_orange_split_0 None diffusion_k_100_t_8_kernel_sknn_include_self_False n_outliers=0, perturbation=None
Done with yao_10x_male_orange_split_0 None diffusion_k_15_t_64_kernel_sknn_include_self_False n_outliers=0, perturbation=None
Done with yao_10x_male_orange_split_0 None diffusion_k_100_t_64_kernel_sknn_include_self_False n_outliers=0, perturbation=None
Done with yao_10x_male_orange_split_0 None eff_res_corrected_True_weighted_False_k_15_disconnect_True n_outliers=0, perturbation=None
Done with yao_10x_male_orange_split_0 None eff_res_corrected_True_weighted_False_k_100_disconnect_True n_outliers=0, perturbation=None
[0, 2, 1, 1, 0, 1]
Done with yao_10x_male_orange_split_1 None diffusion_k_15_t_8_kernel_sknn_include_self_False n_outliers=0, perturbation=None
Done with yao_10x_male_orange_split_1 None diffusion_k_100_t_8_kernel_sknn_include_

In [ ]:
dist = {}
dist['clusters'] = dist1['clusters']
dist['idx'] = [a for i in range(10) for a in globals()[f"dist{i}"]['idx']]
dist['density'] = np.array([a for i in range(10) for a in globals()[f"dist{i}"]['density']])
save_dict(dist, os.path.join(get_path("data"),"dist_bottleneck_newsplit_std_scale", f"dist_10x_male_orange_splits_scale_4.pkl"))

## *3

In [ ]:
geq_maxs_fe = []
for i in range(5):
    dataset = f"yao_10x_female_orange_split_{i}"
    all_res = load_multiple_res(datasets=dataset, n=None, embd_dims=None, sigmas=None, distances=distances, seeds=0, root_path=root_path, n_threads=1)
    geq_maxs = get_life_geq_scale(all_res, threshold_conditions, 3)
    print(geq_maxs)
    geq_maxs_fe.append(geq_maxs)
    globals()[f"dist{i}"] = distribution_split(all_res, cell_group_name = dataset, clusters = cluster_orange_female, full_d = full_d_female, mask = mask_orange_female, geq_maxs = geq_maxs, total_num = 5)
    save_dict(globals()[f"dist{i}"], os.path.join(get_path("data"),"dist_bottleneck_newsplit_std_scale", f"dist_10x_female_orange_split_{i}_scale_3.pkl"))

Done with yao_10x_female_orange_split_0 None diffusion_k_15_t_8_kernel_sknn_include_self_False n_outliers=0, perturbation=None
Done with yao_10x_female_orange_split_0 None diffusion_k_100_t_8_kernel_sknn_include_self_False n_outliers=0, perturbation=None
Done with yao_10x_female_orange_split_0 None diffusion_k_15_t_64_kernel_sknn_include_self_False n_outliers=0, perturbation=None
Done with yao_10x_female_orange_split_0 None diffusion_k_100_t_64_kernel_sknn_include_self_False n_outliers=0, perturbation=None
Done with yao_10x_female_orange_split_0 None eff_res_corrected_True_weighted_False_k_15_disconnect_True n_outliers=0, perturbation=None
Done with yao_10x_female_orange_split_0 None eff_res_corrected_True_weighted_False_k_100_disconnect_True n_outliers=0, perturbation=None
[1, 2, 2, 0, 0, 4]
Done with yao_10x_female_orange_split_1 None diffusion_k_15_t_8_kernel_sknn_include_self_False n_outliers=0, perturbation=None
Done with yao_10x_female_orange_split_1 None diffusion_k_100_t_8_kern

In [ ]:
dist = {}
dist['clusters'] = dist1['clusters']
dist['idx'] = [a for i in range(5) for a in globals()[f"dist{i}"]['idx']]
dist['density'] = np.array([a for i in range(5) for a in globals()[f"dist{i}"]['density']])
save_dict(dist, os.path.join(get_path("data"),"dist_bottleneck_newsplit_std_scale", f"dist_10x_female_orange_splits_scale_4.pkl"))

In [ ]:
geq_maxs_ma = []
for i in range(10):
    dataset = f"yao_10x_male_orange_split_{i}"
    all_res = load_multiple_res(datasets=dataset, n=None, embd_dims=None, sigmas=None, distances=distances, seeds=0, root_path=root_path, n_threads=1)
    geq_maxs = get_life_geq_scale(all_res, threshold_conditions, 3)
    print(geq_maxs)
    geq_maxs_ma.append(geq_maxs)
    globals()[f"dist{i}"] = distribution_split(all_res, cell_group_name = dataset, clusters = cluster_orange_male, full_d = full_d_male, mask = mask_orange_male, geq_maxs = geq_maxs, total_num = 10)
    save_dict(globals()[f"dist{i}"], os.path.join(get_path("data"),"dist_bottleneck_newsplit_std_scale", f"dist_10x_male_orange_split_{i}_scale_3.pkl"))

Done with yao_10x_male_orange_split_0 None diffusion_k_15_t_8_kernel_sknn_include_self_False n_outliers=0, perturbation=None
Done with yao_10x_male_orange_split_0 None diffusion_k_100_t_8_kernel_sknn_include_self_False n_outliers=0, perturbation=None
Done with yao_10x_male_orange_split_0 None diffusion_k_15_t_64_kernel_sknn_include_self_False n_outliers=0, perturbation=None
Done with yao_10x_male_orange_split_0 None diffusion_k_100_t_64_kernel_sknn_include_self_False n_outliers=0, perturbation=None
Done with yao_10x_male_orange_split_0 None eff_res_corrected_True_weighted_False_k_15_disconnect_True n_outliers=0, perturbation=None
Done with yao_10x_male_orange_split_0 None eff_res_corrected_True_weighted_False_k_100_disconnect_True n_outliers=0, perturbation=None
[2, 3, 2, 2, 2, 3]
Done with yao_10x_male_orange_split_1 None diffusion_k_15_t_8_kernel_sknn_include_self_False n_outliers=0, perturbation=None
Done with yao_10x_male_orange_split_1 None diffusion_k_100_t_8_kernel_sknn_include_

In [ ]:
dist = {}
dist['clusters'] = dist1['clusters']
dist['idx'] = [a for i in range(10) for a in globals()[f"dist{i}"]['idx']]
dist['density'] = np.array([a for i in range(10) for a in globals()[f"dist{i}"]['density']])
save_dict(dist, os.path.join(get_path("data"),"dist_bottleneck_newsplit_std_scale", f"dist_10x_male_orange_splits_scale_4.pkl"))

# 90 percentile

In [ ]:
threshold_conditions = []

for full_dist in full_dists:
    bdists = [] 
    for i, dataset in enumerate(sex_data):
        bdist = load_dict(os.path.join(get_path("data"),f"{dataset}/bottleneck_dists_split_quarter_{full_dist}.pkl"))["bottleneck"]
        bdists.append(bdist)
    bdists = np.array(bdists)
    bdists_flat = bdists[np.triu_indices_from(bdists, k=1)]    
    print(len(bdists_flat))
    print(np.percentile(bdists_flat, 90))
    threshold_conditions.append(np.percentile(bdists_flat, 90))

105
0.31193000000000004
105
0.15388139999999995
105
0.05019692000000004
105
0.00910398
105
0.00181395
105
2.9180160000000023e-05


In [ ]:
[float("{:.8f}".format(2*i)) for i in threshold_conditions]

[0.62386, 0.3077628, 0.10039384, 0.01820796, 0.0036279, 5.836e-05]

In [ ]:
geq_maxss0 = []
for i in range(5):
    dataset = f"yao_10x_female_orange_split_{i}"
    all_res = load_multiple_res(datasets=dataset, n=None, embd_dims=None, sigmas=None, distances=distances, seeds=0, root_path=root_path, n_threads=1)
    geq_maxs = get_life_geq_scale(all_res, threshold_conditions, 2)
    print(geq_maxs)
    geq_maxss0.append(geq_maxs)
    globals()[f"dist{i}"] = distribution_split(all_res, cell_group_name = dataset, clusters = cluster_orange_female, full_d = full_d_female, mask = mask_orange_female, geq_maxs = geq_maxs, total_num = 5)
    save_dict(globals()[f"dist{i}"], os.path.join(get_path("data"),"dist_bottleneck_newsplit_90", f"dist_10x_female_orange_split_{i}.pkl"))

Done with yao_10x_female_orange_split_0 None diffusion_k_15_t_8_kernel_sknn_include_self_False n_outliers=0, perturbation=None
Done with yao_10x_female_orange_split_0 None diffusion_k_100_t_8_kernel_sknn_include_self_False n_outliers=0, perturbation=None
Done with yao_10x_female_orange_split_0 None diffusion_k_15_t_64_kernel_sknn_include_self_False n_outliers=0, perturbation=None
Done with yao_10x_female_orange_split_0 None diffusion_k_100_t_64_kernel_sknn_include_self_False n_outliers=0, perturbation=None
Done with yao_10x_female_orange_split_0 None eff_res_corrected_True_weighted_False_k_15_disconnect_True n_outliers=0, perturbation=None
Done with yao_10x_female_orange_split_0 None eff_res_corrected_True_weighted_False_k_100_disconnect_True n_outliers=0, perturbation=None
[0, 2, 0, 0, 0, 1]
Done with yao_10x_female_orange_split_1 None diffusion_k_15_t_8_kernel_sknn_include_self_False n_outliers=0, perturbation=None
Done with yao_10x_female_orange_split_1 None diffusion_k_100_t_8_kern

In [ ]:
geq_maxss0

[[0, 2, 0, 0, 0, 1],
 [1, 2, 1, 1, 0, 0],
 [0, 0, 0, 1, 0, 0],
 [1, 1, 0, 0, 0, 0],
 [1, 1, 0, 0, 0, 0]]

In [ ]:
dist = {}
dist['clusters'] = dist1['clusters']
dist['idx'] = [a for i in range(5) for a in globals()[f"dist{i}"]['idx']]
dist['density'] = np.array([a for i in range(5) for a in globals()[f"dist{i}"]['density']])
save_dict(dist, os.path.join(get_path("data"),"dist_bottleneck_newsplit_90", f"dist_10x_female_orange_splits.pkl"))

In [ ]:
geq_maxss1 = []
for i in range(10):
    dataset = f"yao_10x_male_orange_split_{i}"
    all_res = load_multiple_res(datasets=dataset, n=None, embd_dims=None, sigmas=None, distances=distances, seeds=0, root_path=root_path, n_threads=1)
    geq_maxs = get_life_geq_scale(all_res, threshold_conditions, 2)
    print(geq_maxs)
    geq_maxss1.append(geq_maxs)
    globals()[f"dist{i}"] = distribution_split(all_res, cell_group_name = dataset, clusters = cluster_orange_male, full_d = full_d_male, mask = mask_orange_male, geq_maxs = geq_maxs, total_num = 10)
    save_dict(globals()[f"dist{i}"], os.path.join(get_path("data"),"dist_bottleneck_newsplit_90", f"dist_10x_male_orange_split_{i}.pkl"))

Done with yao_10x_male_orange_split_0 None diffusion_k_15_t_8_kernel_sknn_include_self_False n_outliers=0, perturbation=None
Done with yao_10x_male_orange_split_0 None diffusion_k_100_t_8_kernel_sknn_include_self_False n_outliers=0, perturbation=None
Done with yao_10x_male_orange_split_0 None diffusion_k_15_t_64_kernel_sknn_include_self_False n_outliers=0, perturbation=None
Done with yao_10x_male_orange_split_0 None diffusion_k_100_t_64_kernel_sknn_include_self_False n_outliers=0, perturbation=None
Done with yao_10x_male_orange_split_0 None eff_res_corrected_True_weighted_False_k_15_disconnect_True n_outliers=0, perturbation=None
Done with yao_10x_male_orange_split_0 None eff_res_corrected_True_weighted_False_k_100_disconnect_True n_outliers=0, perturbation=None
[1, 2, 1, 0, 0, 1]
Done with yao_10x_male_orange_split_1 None diffusion_k_15_t_8_kernel_sknn_include_self_False n_outliers=0, perturbation=None
Done with yao_10x_male_orange_split_1 None diffusion_k_100_t_8_kernel_sknn_include_

In [ ]:
dist = {}
dist['clusters'] = dist1['clusters']
dist['idx'] = [a for i in range(10) for a in globals()[f"dist{i}"]['idx']]
dist['density'] = np.array([a for i in range(10) for a in globals()[f"dist{i}"]['density']])
save_dict(dist, os.path.join(get_path("data"),"dist_bottleneck_newsplit_90", f"dist_10x_male_orange_splits.pkl"))

In [ ]:
np.array(geq_maxss1).sum(0)

array([12, 15,  8,  2,  1, 21])

In [ ]:
np.array(geq_maxss0).sum(0)

array([3, 6, 1, 2, 0, 1])

# new 2* median, (by matrix)

In [ ]:
threshold_conditions = []

for full_dist in full_dists:
    bdists = [] 
    for i, dataset in enumerate(sex_data):
        bdist = load_dict(os.path.join(get_path("data"),f"{dataset}/bottleneck_dists_split_quarter_{full_dist}.pkl"))["bottleneck"]
        bdists.append(bdist)
    bdists = np.array(bdists)
    bdists_flat = bdists[np.triu_indices_from(bdists, k=1)]    

    threshold_conditions.append(np.median(bdists_flat))

In [ ]:
geq_maxss0 = []
for i in range(5):
    dataset = f"yao_10x_female_orange_split_{i}"
    all_res = load_multiple_res(datasets=dataset, n=None, embd_dims=None, sigmas=None, distances=distances, seeds=0, root_path=root_path, n_threads=1)
    geq_maxs = get_life_geq_scale(all_res, threshold_conditions, 2)
    print(geq_maxs)
    geq_maxss0.append(geq_maxs)
    globals()[f"dist{i}"] = distribution_split(all_res, cell_group_name = dataset, clusters = cluster_orange_female, full_d = full_d_female, mask = mask_orange_female, geq_maxs = geq_maxs, total_num = 5)
    save_dict(globals()[f"dist{i}"], os.path.join(get_path("data"),"dist_bottleneck_newsplit_median_2_mat", f"dist_10x_female_orange_split_{i}.pkl"))

Done with yao_10x_female_orange_split_0 None diffusion_k_15_t_8_kernel_sknn_include_self_False n_outliers=0, perturbation=None
Done with yao_10x_female_orange_split_0 None diffusion_k_100_t_8_kernel_sknn_include_self_False n_outliers=0, perturbation=None
Done with yao_10x_female_orange_split_0 None diffusion_k_15_t_64_kernel_sknn_include_self_False n_outliers=0, perturbation=None
Done with yao_10x_female_orange_split_0 None diffusion_k_100_t_64_kernel_sknn_include_self_False n_outliers=0, perturbation=None
Done with yao_10x_female_orange_split_0 None eff_res_corrected_True_weighted_False_k_15_disconnect_True n_outliers=0, perturbation=None
Done with yao_10x_female_orange_split_0 None eff_res_corrected_True_weighted_False_k_100_disconnect_True n_outliers=0, perturbation=None
[1, 2, 2, 0, 2, 4]
Done with yao_10x_female_orange_split_1 None diffusion_k_15_t_8_kernel_sknn_include_self_False n_outliers=0, perturbation=None
Done with yao_10x_female_orange_split_1 None diffusion_k_100_t_8_kern

In [ ]:
dist = {}
dist['clusters'] = dist1['clusters']
dist['idx'] = [a for i in range(5) for a in globals()[f"dist{i}"]['idx']]
dist['density'] = np.array([a for i in range(5) for a in globals()[f"dist{i}"]['density']])
save_dict(dist, os.path.join(get_path("data"),"dist_bottleneck_newsplit_median_2_mat", f"dist_10x_female_orange_splits.pkl"))

In [ ]:
geq_maxss1 = []
for i in range(10):
    dataset = f"yao_10x_male_orange_split_{i}"
    all_res = load_multiple_res(datasets=dataset, n=None, embd_dims=None, sigmas=None, distances=distances, seeds=0, root_path=root_path, n_threads=1)
    geq_maxs = get_life_geq_scale(all_res, threshold_conditions, 2)
    print(geq_maxs)
    geq_maxss1.append(geq_maxs)
    globals()[f"dist{i}"] = distribution_split(all_res, cell_group_name = dataset, clusters = cluster_orange_male, full_d = full_d_male, mask = mask_orange_male, geq_maxs = geq_maxs, total_num = 10)
    save_dict(globals()[f"dist{i}"], os.path.join(get_path("data"),"dist_bottleneck_newsplit_median_2_mat", f"dist_10x_male_orange_split_{i}.pkl"))

Done with yao_10x_male_orange_split_0 None diffusion_k_15_t_8_kernel_sknn_include_self_False n_outliers=0, perturbation=None
Done with yao_10x_male_orange_split_0 None diffusion_k_100_t_8_kernel_sknn_include_self_False n_outliers=0, perturbation=None
Done with yao_10x_male_orange_split_0 None diffusion_k_15_t_64_kernel_sknn_include_self_False n_outliers=0, perturbation=None
Done with yao_10x_male_orange_split_0 None diffusion_k_100_t_64_kernel_sknn_include_self_False n_outliers=0, perturbation=None
Done with yao_10x_male_orange_split_0 None eff_res_corrected_True_weighted_False_k_15_disconnect_True n_outliers=0, perturbation=None
Done with yao_10x_male_orange_split_0 None eff_res_corrected_True_weighted_False_k_100_disconnect_True n_outliers=0, perturbation=None
[2, 3, 3, 3, 5, 3]
Done with yao_10x_male_orange_split_1 None diffusion_k_15_t_8_kernel_sknn_include_self_False n_outliers=0, perturbation=None
Done with yao_10x_male_orange_split_1 None diffusion_k_100_t_8_kernel_sknn_include_

In [ ]:
dist = {}
dist['clusters'] = dist1['clusters']
dist['idx'] = [a for i in range(10) for a in globals()[f"dist{i}"]['idx']]
dist['density'] = np.array([a for i in range(10) for a in globals()[f"dist{i}"]['density']])
save_dict(dist, os.path.join(get_path("data"),"dist_bottleneck_newsplit_median_2_mat", f"dist_10x_male_orange_splits.pkl"))